In [ ]:
# 0. Auto-install missing packages (safe to run every time)
# Uses the SAME Python that this notebook/script is running on, so it fixes
# "ModuleNotFoundError: No module named 'seaborn'" even when pasted into one cell.
import sys, subprocess, importlib, importlib.util

def ensure(packages):
    missing = [pip_name for pip_name, module in packages
               if importlib.util.find_spec(module) is None]
    if missing:
        print("Installing:", missing)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])
        importlib.invalidate_caches()
    else:
        print("All packages already installed.")

ensure([("seaborn", "seaborn"), ("ucimlrepo", "ucimlrepo"),
        ("scikit-learn", "sklearn"), ("matplotlib", "matplotlib"),
        ("pandas", "pandas"), ("numpy", "numpy")])
# (optional) XGBoost: ensure([("xgboost", "xgboost")])


In [ ]:
# 1. Imports
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.feature_selection import VarianceThreshold, SelectKBest, f_classif
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (classification_report, confusion_matrix,
                             accuracy_score, f1_score,
                             adjusted_rand_score, silhouette_score)
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering

os.makedirs("figures", exist_ok=True)
RS = 42
sns.set_style("whitegrid")


In [ ]:
# 2. Choose the dataset file
# A file-chooser window opens (it may appear BEHIND your browser - check the taskbar).
# You can select ANY of these:
#   (a) the downloaded UCI file as it is: gene+expression+cancer+rna+seq.zip  (or the .tar.gz inside it)
#   (b) data.csv  -> then you will be asked to pick labels.csv as well
#   (c) your own .csv that already has a label column named Class / label / target
# Press Cancel in the window to download the dataset automatically from UCI instead.
#
# To skip the window, paste the full path(s) below, e.g.
#   DATA_PATH = r"C:\Users\you\Downloads\TCGA-PANCAN-HiSeq-801x20531.tar.gz"
DATA_PATH = ""
LABELS_PATH = ""      # only needed if DATA_PATH is a data.csv without labels

import tarfile, zipfile, io

LABEL_NAMES = ("class", "label", "labels", "target", "cancer_type", "type")


def pick_file(title, filetypes):
    """Open a file-chooser window (works when Jupyter runs on your own PC)."""
    try:
        import tkinter as tk
        from tkinter import filedialog
        root = tk.Tk()
        root.withdraw()
        root.attributes("-topmost", True)
        path = filedialog.askopenfilename(title=title, filetypes=filetypes)
        root.destroy()
        return path
    except Exception as e:
        print("File dialog not available:", e)
        return ""


def read_table(src):
    """Read a csv; if the first column holds sample IDs, use it as the index."""
    df = pd.read_csv(src)
    first = df.columns[0]
    if str(first).lower().startswith("unnamed") or df[first].dtype == object:
        if str(first).lower() not in LABEL_NAMES:
            df = df.set_index(first)
    return df


def _read_tar(tar):
    """Read data.csv + labels.csv from an open tar archive."""
    members = tar.getmembers()
    dm = next(m for m in members if m.name.endswith("data.csv"))
    lm = next(m for m in members if m.name.endswith("labels.csv"))
    return read_table(tar.extractfile(dm)), read_table(tar.extractfile(lm)).iloc[:, 0]


def load_dataset():
    path = DATA_PATH or pick_file(
        "Select data file (.zip / .tar.gz / data.csv)",
        [("Dataset files", "*.zip *.csv *.gz *.tgz *.tar"), ("All files", "*.*")])

    # Nothing chosen -> download from UCI
    if not path:
        print("No file chosen -> downloading from the UCI repository...")
        from ucimlrepo import fetch_ucirepo
        d = fetch_ucirepo(id=401)
        return d.data.features, d.data.targets.iloc[:, 0]

    print("Using file:", path)

    low = path.lower()

    # (a1) the UCI download: a .zip (usually containing the .tar.gz, or the csv files directly)
    if low.endswith(".zip"):
        with zipfile.ZipFile(path) as zf:
            names = zf.namelist()
            tar_name = next((n for n in names if n.lower().endswith((".tar.gz", ".tgz", ".tar"))), None)
            if tar_name:
                with tarfile.open(fileobj=io.BytesIO(zf.read(tar_name))) as tar:
                    return _read_tar(tar)
            dn = next(n for n in names if n.endswith("data.csv"))
            ln = next(n for n in names if n.endswith("labels.csv"))
            return read_table(zf.open(dn)), read_table(zf.open(ln)).iloc[:, 0]

    # (a2) a .tar.gz archive
    if low.endswith((".tar.gz", ".tgz", ".tar")):
        with tarfile.open(path) as tar:
            return _read_tar(tar)

    # (b)/(c) csv
    Xd = read_table(path)
    for c in Xd.columns:
        if str(c).lower() in LABEL_NAMES:        # label column inside the file
            return Xd.drop(columns=c), Xd[c]
    lp = LABELS_PATH or pick_file("Select labels file (labels.csv)",
                                  [("CSV files", "*.csv"), ("All files", "*.*")])
    if not lp:
        raise ValueError("No label column found and no labels file selected.")
    yd = read_table(lp).iloc[:, 0]
    if Xd.index.equals(yd.index) or set(Xd.index) == set(yd.index):
        yd = yd.loc[Xd.index]
    return Xd, yd


X, y_raw = load_dataset()
X = X.select_dtypes(include="number").reset_index(drop=True)   # keep numeric gene columns only
y_raw = pd.Series(y_raw).reset_index(drop=True)

print("Shape:", X.shape)
print("Missing values:", int(X.isna().sum().sum()))
print(y_raw.value_counts())

le = LabelEncoder()
y = le.fit_transform(y_raw)
classes = le.classes_
print("Classes:", classes)


In [ ]:
# 3. EDA: class distribution
plt.figure(figsize=(6, 4))
sns.countplot(x=y_raw, order=y_raw.value_counts().index)
plt.title("Samples per cancer type")
plt.xlabel("Cancer type"); plt.ylabel("Count")
plt.tight_layout(); plt.savefig("figures/01_class_distribution.png", dpi=200); plt.show()


In [ ]:
# 4. Unsupervised view: PCA (no labels used to build the components)
X_scaled = StandardScaler().fit_transform(X)
pca_full = PCA(n_components=50, random_state=RS).fit(X_scaled)
pcs = pca_full.transform(X_scaled)

# 4a. Explained variance
plt.figure(figsize=(6, 4))
plt.plot(np.cumsum(pca_full.explained_variance_ratio_) * 100, marker="o", ms=3)
plt.xlabel("Number of principal components"); plt.ylabel("Cumulative variance explained (%)")
plt.title("PCA: cumulative explained variance")
plt.tight_layout(); plt.savefig("figures/02_pca_variance.png", dpi=200); plt.show()

# 4b. 2D scatter coloured by true cancer type
plt.figure(figsize=(7, 5))
sns.scatterplot(x=pcs[:, 0], y=pcs[:, 1], hue=y_raw.values, palette="deep", s=25)
plt.xlabel(f"PC1 ({pca_full.explained_variance_ratio_[0]*100:.1f}%)")
plt.ylabel(f"PC2 ({pca_full.explained_variance_ratio_[1]*100:.1f}%)")
plt.title("PCA of gene expression profiles"); plt.legend(title="Cancer type")
plt.tight_layout(); plt.savefig("figures/03_pca_scatter.png", dpi=200); plt.show()


In [ ]:
# 5. Clustering (unsupervised) on the top 20 PCs
Z = pcs[:, :20]
km = KMeans(n_clusters=5, n_init=10, random_state=RS).fit(Z)
hc = AgglomerativeClustering(n_clusters=5, linkage="ward").fit(Z)

print("K-means      ARI:", round(adjusted_rand_score(y, km.labels_), 3),
      "| silhouette:", round(silhouette_score(Z, km.labels_), 3))
print("Hierarchical ARI:", round(adjusted_rand_score(y, hc.labels_), 3),
      "| silhouette:", round(silhouette_score(Z, hc.labels_), 3))

# Cluster vs true class table
ct = pd.crosstab(pd.Series(y_raw.values, name="True cancer type"),
                 pd.Series(km.labels_, name="K-means cluster"))
plt.figure(figsize=(6, 4))
sns.heatmap(ct, annot=True, fmt="d", cmap="Blues")
plt.title("K-means clusters vs true cancer type")
plt.tight_layout(); plt.savefig("figures/04_kmeans_vs_true.png", dpi=200); plt.show()


In [ ]:
# 6. Train/test split (stratified keeps class proportions)
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RS)


In [ ]:
# 7. Supervised models
# Feature selection + scaling are inside the Pipeline, so they are fitted on
# training folds only (no data leakage). 20,531 genes -> top 500 by ANOVA F-test.
def make_pipe(model):
    return Pipeline([
        ("var", VarianceThreshold(0.0)),          # drop constant genes
        ("scale", StandardScaler()),
        ("select", SelectKBest(f_classif, k=500)),
        ("clf", model),
    ])

models = {
    "Logistic Regression": LogisticRegression(max_iter=2000, class_weight="balanced"),
    "SVM (RBF)": SVC(kernel="rbf", class_weight="balanced", random_state=RS),
    "Random Forest": RandomForestClassifier(n_estimators=300, class_weight="balanced",
                                            random_state=RS, n_jobs=-1),
}
# Optional: XGBoost (uncomment if installed)
# from xgboost import XGBClassifier
# models["XGBoost"] = XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
#                                   eval_metric="mlogloss", random_state=RS)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RS)
rows, fitted = [], {}
for name, model in models.items():
    pipe = make_pipe(model)
    cv_f1 = cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="f1_macro", n_jobs=-1)
    pipe.fit(X_tr, y_tr)
    pred = pipe.predict(X_te)
    fitted[name] = pipe
    rows.append({
        "Model": name,
        "CV macro-F1 (mean)": round(cv_f1.mean(), 4),
        "CV macro-F1 (std)": round(cv_f1.std(), 4),
        "Test accuracy": round(accuracy_score(y_te, pred), 4),
        "Test macro-F1": round(f1_score(y_te, pred, average="macro"), 4),
    })

results = pd.DataFrame(rows).set_index("Model")
print(results)
results.to_csv("figures/model_comparison.csv")


In [ ]:
# 8. Best model: classification report + confusion matrix
best_name = results["Test macro-F1"].idxmax()
best = fitted[best_name]
pred = best.predict(X_te)
print(f"Best model: {best_name}\n")
print(classification_report(y_te, pred, target_names=classes))

plt.figure(figsize=(6, 5))
sns.heatmap(confusion_matrix(y_te, pred), annot=True, fmt="d", cmap="Blues",
            xticklabels=classes, yticklabels=classes)
plt.xlabel("Predicted"); plt.ylabel("Actual"); plt.title(f"Confusion matrix: {best_name}")
plt.tight_layout(); plt.savefig("figures/05_confusion_matrix.png", dpi=200); plt.show()


In [ ]:
# 9. Model comparison bar chart
ax = results[["Test accuracy", "Test macro-F1"]].plot(kind="bar", figsize=(7, 4), ylim=(0.9, 1.01))
plt.title("Model comparison (test set)"); plt.xticks(rotation=15); plt.ylabel("Score")
plt.tight_layout(); plt.savefig("figures/06_model_comparison.png", dpi=200); plt.show()


In [ ]:
# 10. Important genes (Random Forest on the selected genes)
rf_pipe = fitted["Random Forest"]
kept_after_var = X.columns[rf_pipe.named_steps["var"].get_support()]
selected_genes = kept_after_var[rf_pipe.named_steps["select"].get_support()]
importances = pd.Series(rf_pipe.named_steps["clf"].feature_importances_,
                        index=selected_genes).nlargest(15)

plt.figure(figsize=(6, 5))
importances[::-1].plot(kind="barh")
plt.title("Top 15 genes by Random Forest importance")
plt.xlabel("Importance")
plt.tight_layout(); plt.savefig("figures/07_top_genes.png", dpi=200); plt.show()
print(importances)
# NOTE: gene names in this UCI file are masked (gene_XX). To name them, map the
# column order to the original TCGA PANCAN probe list from the Synapse submission.


In [ ]:
# 11. Optional: show that few genes are enough (accuracy vs number of genes)
ks = [10, 25, 50, 100, 250, 500, 1000, 2000]
accs = []
for k in ks:
    pipe = Pipeline([
        ("var", VarianceThreshold(0.0)),
        ("scale", StandardScaler()),
        ("select", SelectKBest(f_classif, k=k)),
        ("clf", LogisticRegression(max_iter=2000, class_weight="balanced")),
    ])
    accs.append(cross_val_score(pipe, X_tr, y_tr, cv=cv, scoring="f1_macro", n_jobs=-1).mean())

plt.figure(figsize=(6, 4))
plt.plot(ks, accs, marker="o"); plt.xscale("log")
plt.xlabel("Number of genes used"); plt.ylabel("CV macro-F1")
plt.title("Performance vs number of genes (Logistic Regression)")
plt.tight_layout(); plt.savefig("figures/08_genes_vs_performance.png", dpi=200); plt.show()
